## Read IBTrACS data file for TC tracks and plot the locations of all TC genesis over range of months

### NCSU Tropical and Large Scale Dynamics
### Code by A. Aiyyer with help from Claude AI
### AI Usage declaration:
- Claude Code used to debug code and improve efficiency
  

In [ ]:
from matplotlib.dates import num2date,date2num
import xarray as xr
import numpy as np
import datetime as dt
import cftime as cf
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import cartopy.feature as cfeature
from   calendar import monthrange
import pandas as pd

## some basic settings
## where is the ibtracs file?

In [ ]:

dataDir  = "../data/"
filename = "IBTrACS.ALL.v04r01.nc"

file = dataDir+filename

try:
    ds = xr.open_dataset(file)
    print ("Ibtracs file found and opened")
except:
    print ("file not found. quitting code")
    
    
# read the first record for each TC
time  = ds.time[:,0]
basin = ds.basin[:,0]
name  = ds.name

# define the function for the plot

In [ ]:
# Plot settings
def plot_locations(minlon,laxlon,minlat,maxlat,lon,lat):
    data_crs = ccrs.PlateCarree()
    fig = plt.figure(figsize=(15,7))
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())
    ax.set_extent([minlon,maxlon,minlat,maxlat], crs=ccrs.PlateCarree())
    #ax.set_global()
    ax.coastlines(linewidths=0.5, color='grey')
    ax.add_feature(cfeature.LAND,color='lightgrey')
    #ax.spines['geo'].set_edgecolor('black')

# marker size is controlled by s
    ax.scatter(lon,lat,s=5,transform=data_crs)
    return ax
def plotTitle(ax,titleString):
        ax.set_title(titleString, fontsize=16)


## plot all TCs over the globe for a selected range of years


In [ ]:

# plot extent is global for this one

minlon = -180.
maxlon =  180.
minlat =  -40.
maxlat =   40.

yearStart = 1970
yearEnd   = 2025

tsub = time.where((time["time.year"] >= yearStart)&(time["time.year"] <= yearEnd) , drop=True)


# plot the locations

ax=plot_locations(minlon,maxlon,minlat,maxlat,tsub.lon,tsub.lat)
titleString = "TC Formation Locations: " + str(yearStart) + "--" + str(yearEnd) + " All Months" 
plotTitle(ax,titleString)

## plot all TCs over the globe for a selected range of years And Months


In [ ]:
#--------------------------------------------------
minlon = -180.
maxlon =  180.
minlat =  -40.
maxlat =   40.

monthStart = 6
monthEnd   = 6
#--------------------------------------------------

tsub = time.where((time["time.year"] >= yearStart)&(time["time.year"] <= yearEnd) , drop=True)
# further subset for the month range
tsub = tsub.where((tsub["time.month"] >= monthStart)&(tsub["time.month"] <= monthEnd) , drop=True)

# plot the locations
ax=plot_locations(minlon,maxlon,minlat,maxlat,tsub.lon,tsub.lat)
titleString = "Genesis: " + str(yearStart) + "--" + str(yearEnd) + " Months: " + str(monthStart) +"--"+ str(monthEnd)
plotTitle(ax,titleString)


## Animate global TC genesis locations, month by month

In [ ]:
import matplotlib.animation as animation
from IPython.display import HTML

# domain / year range (reuse settings from above; edit as needed)
minlon = -180.
maxlon =  180.
minlat =  -40.
maxlat =   40.

yearStart = 1970
yearEnd   = 2025

months = np.arange(1, 13)

# genesis times restricted to the year range
tsub_years = time.where((time["time.year"] >= yearStart) & (time["time.year"] <= yearEnd), drop=True)

data_crs = ccrs.PlateCarree()
fig = plt.figure(figsize=(15, 7))
ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())

def init():
    ax.set_extent([minlon, maxlon, minlat, maxlat], crs=data_crs)
    ax.coastlines(linewidths=0.5, color='grey')
    ax.add_feature(cfeature.LAND, color='lightgrey')
    return []

def update(month):
    ax.clear()
    ax.set_extent([minlon, maxlon, minlat, maxlat], crs=data_crs)
    ax.coastlines(linewidths=0.5, color='grey')
    ax.add_feature(cfeature.LAND, color='lightgrey')

    tm = tsub_years.where(tsub_years["time.month"] == month, drop=True)
    ax.scatter(tm.lon, tm.lat, s=8, color='red', transform=data_crs)

    titleString = ("TC Genesis Locations \u2014 Month " + str(month).zfill(2) +
                   "  (" + str(yearStart) + "\u2013" + str(yearEnd) + ")")
    ax.set_title(titleString, fontsize=14)
    return []

anim = animation.FuncAnimation(fig, update, frames=months, init_func=init,
                                interval=800, blit=False, repeat=True)
plt.close(fig)
HTML(anim.to_jshtml())


## Animate global TC tracks, grouped by genesis month

In [ ]:
# Read the FULL track lat/lon/time (all records per storm, not just genesis)
lat_all  = ds.lat
lon_all  = ds.lon
time_all = ds.time

# genesis month/year per storm, from the first-record time array read earlier
genesis_month = time["time.month"]
genesis_year  = time["time.year"]

fig2 = plt.figure(figsize=(15, 7))
ax2 = fig2.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())

def init2():
    ax2.set_extent([minlon, maxlon, minlat, maxlat], crs=data_crs)
    ax2.coastlines(linewidths=0.5, color='grey')
    ax2.add_feature(cfeature.LAND, color='lightgrey')
    return []

def update2(month):
    ax2.clear()
    ax2.set_extent([minlon, maxlon, minlat, maxlat], crs=data_crs)
    ax2.coastlines(linewidths=0.5, color='grey')
    ax2.add_feature(cfeature.LAND, color='lightgrey')

    # storms whose genesis fell in this month, within the year range
    mask = ((genesis_month == month) &
            (genesis_year >= yearStart) & (genesis_year <= yearEnd))
    storm_idx = np.where(mask.values)[0]

    for idx in storm_idx:
        storm_lon = lon_all[idx, :].values
        storm_lat = lat_all[idx, :].values
        valid = ~np.isnan(storm_lon) & ~np.isnan(storm_lat)
        ax2.plot(storm_lon[valid], storm_lat[valid], linewidth=1, transform=data_crs)

    titleString = ("TC Tracks \u2014 Genesis Month " + str(month).zfill(2) +
                   "  (" + str(yearStart) + "\u2013" + str(yearEnd) + ")" +
                   "  [" + str(len(storm_idx)) + " storms]")
    ax2.set_title(titleString, fontsize=14)
    return []

anim2 = animation.FuncAnimation(fig2, update2, frames=months, init_func=init2,
                                 interval=800, blit=False, repeat=True)
plt.close(fig2)
HTML(anim2.to_jshtml())


## Number of NAMED tropical cyclones per year, globally (1980–2025)

In [ ]:
yearStart = 1980
yearEnd   = 2025

# Decode the storm name variable and build a mask for NAMED storms only.
# Unnamed storms in IBTrACS are labeled "NOT_NAMED" (sometimes "UNNAMED" or blank).
name_arr = name.values
if name_arr.dtype.kind == 'S':
    name_arr = np.char.decode(name_arr, 'utf-8')
name_arr = np.array([str(n).strip().upper() for n in name_arr])

named_mask = ~np.isin(name_arr, ['NOT_NAMED', 'UNNAMED', ''])

years = time["time.year"].values
year_range = np.arange(yearStart, yearEnd + 1)

# restrict to named storms only
years_named = years[named_mask]

counts_global = np.array([np.sum(years_named == y) for y in year_range])

plt.figure(figsize=(12, 5))
plt.plot(year_range, counts_global, marker='o', color='black')
plt.xlabel("Year")
plt.ylabel("Number of Named TCs")
plt.title("Global Named TC Count by Year (" + str(yearStart) + "\u2013" + str(yearEnd) + ")")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
